In [1]:
import logging
import pandas as pd

# ---------------------------------------------------------
# Configure logging
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ---------------------------------------------------------
# Load raw CSV with proper exception handling
# ---------------------------------------------------------
def load_raw_csv(file_path: str) -> pd.DataFrame:
    """
    Load the raw CSV file with logging and safe exception handling.
    """
    logging.info(f"Attempting to load CSV file: {file_path}")

    try:
        df = pd.read_csv(file_path)
        logging.info(f"CSV loaded successfully. Shape: {df.shape}")
        return df

    except FileNotFoundError as e:
        logging.error(f"File not found: {file_path}")
        raise e

    except pd.errors.ParserError as e:
        logging.error(f"CSV parsing error: {e}")
        raise e

    except Exception as e:
        logging.error(f"Unexpected error occurred: {e}")
        raise e

# ---------------------------------------------------------
# Main pipeline entry point
# ---------------------------------------------------------
if __name__ == "__main__":
    logging.info("Starting data pipeline...")

    raw_file = "Metro_Interstate_Traffic_Volume.csv"

    df = load_raw_csv(raw_file)

    logging.info("Pipeline completed successfully.")



2026-09-23 23:08:17,213 - INFO - Starting data pipeline...
2026-09-23 23:08:17,214 - INFO - Attempting to load CSV file: Metro_Interstate_Traffic_Volume.csv
2026-09-23 23:08:17,340 - INFO - CSV loaded successfully. Shape: (48204, 9)
2026-09-23 23:08:17,341 - INFO - Pipeline completed successfully.


In [2]:
import logging
import pandas as pd

# ---------------------------------------------------------
# Configure logging
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ---------------------------------------------------------
# Load raw CSV with proper exception handling
# ---------------------------------------------------------
def load_raw_csv(file_path: str) -> pd.DataFrame:
    """
    Load the raw CSV file with logging and safe exception handling.
    """
    logging.info(f"Attempting to load CSV file: {file_path}")

    try:
        df = pd.read_csv(file_path)
        logging.info(f"CSV loaded successfully. Shape: {df.shape}")
        return df

    except FileNotFoundError as e:
        logging.error(f"File not found: {file_path}")
        raise e

    except pd.errors.ParserError as e:
        logging.error(f"CSV parsing error: {e}")
        raise e

    except Exception as e:
        logging.error(f"Unexpected error occurred: {e}")
        raise e

# ---------------------------------------------------------
# Validate data schema
# ---------------------------------------------------------
def validate_schema(df: pd.DataFrame, expected_columns: list) -> None:
    """
    Validate that all expected columns are present in the dataset.
    """
    logging.info("Validating data schema...")

    missing_columns = [col for col in expected_columns if col not in df.columns]

    if missing_columns:
        logging.error(f"Schema validation failed. Missing columns: {missing_columns}")
        raise ValueError(f"Missing columns in dataset: {missing_columns}")

    logging.info("Schema validation successful. All expected columns are present.")

# ---------------------------------------------------------
# Main pipeline entry point
# ---------------------------------------------------------
if __name__ == "__main__":
    logging.info("Starting data pipeline...")

    raw_file = "Metro_Interstate_Traffic_Volume.csv"

    df = load_raw_csv(raw_file)

    expected_columns = [
        "holiday",
        "temp",
        "rain_1h",
        "snow_1h",
        "clouds_all",
        "weather_main",
        "weather_description",
        "date_time",
        "traffic_volume"
    ]

    validate_schema(df, expected_columns)

    logging.info("Pipeline completed successfully.")



2026-09-23 23:08:35,881 - INFO - Starting data pipeline...
2026-09-23 23:08:35,883 - INFO - Attempting to load CSV file: Metro_Interstate_Traffic_Volume.csv
2026-09-23 23:08:35,961 - INFO - CSV loaded successfully. Shape: (48204, 9)
2026-09-23 23:08:35,964 - INFO - Validating data schema...
2026-09-23 23:08:35,964 - INFO - Schema validation successful. All expected columns are present.
2026-09-23 23:08:35,965 - INFO - Pipeline completed successfully.


In [3]:
# ---------------------------------------------------------
# Clean the data
# ---------------------------------------------------------
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the dataset by standardising categories, parsing dates,
    removing duplicates, detecting outliers, and imputing values.
    """
    logging.info("Starting data cleaning...")

    # -----------------------------------------------------
    # 1. Standardise inconsistent categorical values
    # -----------------------------------------------------
    logging.info("Standardising categorical values...")
    df["weather_main"] = df["weather_main"].str.strip().str.lower()
    df["weather_description"] = df["weather_description"].str.strip().str.lower()
    df["holiday"] = df["holiday"].str.strip().str.lower()

    # -----------------------------------------------------
    # 2. Parse and validate date/time fields
    # -----------------------------------------------------
    logging.info("Parsing date_time column...")
    try:
        df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
    except Exception as e:
        logging.error(f"Error parsing date_time: {e}")
        raise e

    # Remove rows where date_time could not be parsed
    invalid_dates = df["date_time"].isna().sum()
    if invalid_dates > 0:
        logging.warning(f"Removing {invalid_dates} rows with invalid date_time values.")
        df = df.dropna(subset=["date_time"])

    # -----------------------------------------------------
    # 3. Remove duplicate rows
    # -----------------------------------------------------
    logging.info("Removing duplicate rows...")
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    logging.info(f"Removed {before - after} duplicate rows.")

    # -----------------------------------------------------
    # 4. Detect impossible values (outliers)
    # -----------------------------------------------------
    logging.info("Detecting impossible sensor values...")

    # Temperature: Kelvin cannot be 0
    impossible_temp = df[df["temp"] <= 0].shape[0]
    if impossible_temp > 0:
        logging.warning(f"Found {impossible_temp} impossible temperature values (<= 0 Kelvin).")

    # Rainfall: physically impossible > 9000 mm
    impossible_rain = df[df["rain_1h"] > 9000].shape[0]
    if impossible_rain > 0:
        logging.warning(f"Found {impossible_rain} impossible rainfall values (> 9000 mm).")

    # -----------------------------------------------------
    # 5. Impute outliers using median
    # -----------------------------------------------------
    logging.info("Imputing outliers using median...")

    temp_median = df[df["temp"] > 0]["temp"].median()
    df.loc[df["temp"] <= 0, "temp"] = temp_median

    rain_median = df[df["rain_1h"] <= 9000]["rain_1h"].median()
    df.loc[df["rain_1h"] > 9000, "rain_1h"] = rain_median

    logging.info("Data cleaning completed successfully.")
    return df

if __name__ == "__main__":
    logging.info("Starting data pipeline...")

    raw_file = "Metro_Interstate_Traffic_Volume.csv"

    df = load_raw_csv(raw_file)

    expected_columns = [
        "holiday",
        "temp",
        "rain_1h",
        "snow_1h",
        "clouds_all",
        "weather_main",
        "weather_description",
        "date_time",
        "traffic_volume"
    ]

    validate_schema(df, expected_columns)

    df = clean_data(df)
    logging.info(f"Data cleaning complete. Final shape: {df.shape}")

    logging.info("Pipeline completed successfully.")

2026-09-23 23:08:49,058 - INFO - Starting data pipeline...
2026-09-23 23:08:49,060 - INFO - Attempting to load CSV file: Metro_Interstate_Traffic_Volume.csv
2026-09-23 23:08:49,159 - INFO - CSV loaded successfully. Shape: (48204, 9)
2026-09-23 23:08:49,161 - INFO - Validating data schema...
2026-09-23 23:08:49,161 - INFO - Schema validation successful. All expected columns are present.
2026-09-23 23:08:49,162 - INFO - Starting data cleaning...
2026-09-23 23:08:49,162 - INFO - Standardising categorical values...
2026-09-23 23:08:49,195 - INFO - Parsing date_time column...
2026-09-23 23:08:49,216 - INFO - Removing duplicate rows...
2026-09-23 23:08:49,247 - INFO - Removed 17 duplicate rows.
2026-09-23 23:08:49,248 - INFO - Detecting impossible sensor values...
2026-09-23 23:08:49,251 - WARNING - Found 10 impossible temperature values (<= 0 Kelvin).
2026-09-23 23:08:49,253 - WARNING - Found 1 impossible rainfall values (> 9000 mm).
2026-09-23 23:08:49,254 - INFO - Imputing outliers using 

In [4]:
# ---------------------------------------------------------
# Clean the data with conditionals and loops (monthly imputation)
# ---------------------------------------------------------
def clean_data_with_monthly_imputation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the dataset using conditionals and loops.
    Impute outliers using monthly medians instead of global medians.
    """
    logging.info("Starting advanced cleaning with monthly imputation...")

    # Ensure date_time is parsed
    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
    df = df.dropna(subset=["date_time"])

    # Extract month for group-by processing
    df["month"] = df["date_time"].dt.month

    # Loop through each month and apply monthly medians
    for month in df["month"].unique():
        logging.info(f"Processing month: {month}")

        # Subset for the month
        month_df = df[df["month"] == month]

        # -----------------------------------------------------
        # Temperature outliers (<= 0 Kelvin)
        # -----------------------------------------------------
        temp_outliers = month_df[month_df["temp"] <= 0].shape[0]
        if temp_outliers > 0:
            logging.warning(f"Month {month}: Found {temp_outliers} impossible temperature values.")

            # Monthly median (only valid temps)
            temp_median = month_df[month_df["temp"] > 0]["temp"].median()

            # Apply imputation using conditionals
            df.loc[(df["month"] == month) & (df["temp"] <= 0), "temp"] = temp_median

        # -----------------------------------------------------
        # Rainfall outliers (> 9000 mm)
        # -----------------------------------------------------
        rain_outliers = month_df[month_df["rain_1h"] > 9000].shape[0]
        if rain_outliers > 0:
            logging.warning(f"Month {month}: Found {rain_outliers} impossible rainfall values.")

            # Monthly median (only valid rainfall)
            rain_median = month_df[month_df["rain_1h"] <= 9000]["rain_1h"].median()

            # Apply imputation using conditionals
            df.loc[(df["month"] == month) & (df["rain_1h"] > 9000), "rain_1h"] = rain_median

    logging.info("Monthly imputation completed successfully.")

    # Remove helper column
    df = df.drop(columns=["month"])

    return df

df = clean_data(df)
df = clean_data_with_monthly_imputation(df)

logging.info(f"Advanced cleaning complete. Final shape: {df.shape}")

2026-09-23 23:21:25,788 - INFO - Starting data cleaning...
2026-09-23 23:21:25,789 - INFO - Standardising categorical values...
2026-09-23 23:21:25,855 - INFO - Parsing date_time column...
2026-09-23 23:21:25,884 - INFO - Removing duplicate rows...
2026-09-23 23:21:25,907 - INFO - Removed 0 duplicate rows.
2026-09-23 23:21:25,908 - INFO - Detecting impossible sensor values...
2026-09-23 23:21:25,913 - INFO - Imputing outliers using median...
2026-09-23 23:21:25,924 - INFO - Data cleaning completed successfully.
2026-09-23 23:21:25,926 - INFO - Starting advanced cleaning with monthly imputation...
2026-09-23 23:21:25,950 - INFO - Processing month: 10
2026-09-23 23:21:25,956 - INFO - Processing month: 11
2026-09-23 23:21:25,959 - INFO - Processing month: 12
2026-09-23 23:21:25,963 - INFO - Processing month: 1
2026-09-23 23:21:25,966 - INFO - Processing month: 2
2026-09-23 23:21:25,969 - INFO - Processing month: 3
2026-09-23 23:21:25,973 - INFO - Processing month: 4
2026-09-23 23:21:25,97

In [15]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create ML-ready features:
    - Time features (hour, day of week, weekend, cyclical encoding)
    - Weather encodings
    - Derived weather indicators
    - Normalised/scaled numerical features
    - Data-driven congestion category
    """

    # Log dataset shape BEFORE feature engineering
    logging.info(f"Feature Engineering START — Shape before: {df.shape[0]} rows, {df.shape[1]} columns")

    # Ensure date_time is datetime (prevents .dt errors)
    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
    df = df.dropna(subset=["date_time"])

    # -----------------------------------------------------
    # TIME FEATURES
    # -----------------------------------------------------
    logging.info("Creating time-based features...")

    df["hour"] = df["date_time"].dt.hour
    df["day_of_week"] = df["date_time"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    # Cyclical encoding for hour
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    # -----------------------------------------------------
    # WEATHER FEATURES
    # -----------------------------------------------------
    logging.info("Encoding weather features...")

    # Encode categorical weather variable
    df["weather_main_encoded"] = df["weather_main"].astype("category").cat.codes

    # Derived indicators
    df["is_rain"] = df["weather_main"].str.contains("rain").astype(int)
    df["is_snow"] = df["weather_main"].str.contains("snow").astype(int)
    df["is_cloudy"] = df["weather_main"].str.contains("cloud").astype(int)

    # -----------------------------------------------------
    # NUMERICAL FEATURES (Scaling)
    # -----------------------------------------------------
    logging.info("Scaling numerical features...")

    # Min-max scaling for temperature
    df["temp_scaled"] = (df["temp"] - df["temp"].min()) / (df["temp"].max() - df["temp"].min())

    # Standard scaling for cloud cover
    df["clouds_scaled"] = (df["clouds_all"] - df["clouds_all"].mean()) / df["clouds_all"].std()

    # -----------------------------------------------------
    # TRAFFIC TARGET (Data-driven congestion category)
    # -----------------------------------------------------
    logging.info("Creating congestion category...")

    # Quartile thresholds — DEBUG only
    q1 = df["traffic_volume"].quantile(0.33)
    q2 = df["traffic_volume"].quantile(0.66)

    logging.debug(f"DEBUG — Congestion thresholds: q1={q1}, q2={q2}")

    def congestion_category(v):
        if v < q1:
            return "low"
        elif v < q2:
            return "medium"
        else:
            return "high"

    df["congestion_category"] = df["traffic_volume"].apply(congestion_category)

    # Log dataset shape AFTER feature engineering
    logging.info(f"Feature Engineering END — Shape after: {df.shape[0]} rows, {df.shape[1]} columns")

    logging.info("Feature engineering completed successfully.")
    return df

import logging
logging.basicConfig(level=logging.INFO)

df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")

# Convert date_time BEFORE feature engineering
df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
df = df.dropna(subset=["date_time"])

df = feature_engineering(df)
df.head()
    

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,hour,...,is_weekend,hour_sin,hour_cos,weather_main_encoded,is_rain,is_snow,is_cloudy,temp_scaled,clouds_scaled,congestion_category
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,9,...,0,7.071068e-01,-0.707107,1,0,0,0,0.929726,-0.239960,high
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,10,...,0,5.000000e-01,-0.866025,1,0,0,0,0.933209,0.657113,medium
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,11,...,0,2.588190e-01,-0.965926,1,0,0,0,0.933918,1.041573,high
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,12,...,0,1.224647e-16,-1.000000,1,0,0,0,0.935692,1.041573,high
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,13,...,0,-2.588190e-01,-0.965926,1,0,0,0,0.938949,0.657113,high


In [14]:
import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    logging.info(f"Feature Engineering START — Shape before: {df.shape}")

    # Ensure datetime
    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
    df = df.dropna(subset=["date_time"])

    # Time features
    df["hour"] = df["date_time"].dt.hour
    df["day_of_week"] = df["date_time"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    # Weather features
    df["weather_main_encoded"] = df["weather_main"].astype("category").cat.codes
    df["is_rain"] = df["weather_main"].str.contains("rain").astype(int)
    df["is_snow"] = df["weather_main"].str.contains("snow").astype(int)
    df["is_cloudy"] = df["weather_main"].str.contains("cloud").astype(int)

    # Numerical scaling
    df["temp_scaled"] = (df["temp"] - df["temp"].min()) / (df["temp"].max() - df["temp"].min())
    df["clouds_scaled"] = (df["clouds_all"] - df["clouds_all"].mean()) / df["clouds_all"].std()

    # Congestion category
    q1 = df["traffic_volume"].quantile(0.33)
    q2 = df["traffic_volume"].quantile(0.66)

    def congestion_category(v):
        if v < q1:
            return "low"
        elif v < q2:
            return "medium"
        else:
            return "high"

    df["congestion_category"] = df["traffic_volume"].apply(congestion_category)

    logging.info(f"Feature Engineering END — Shape after: {df.shape}")
    return df

In [15]:
import matplotlib.pyplot as plt

def visualise_traffic_patterns(df: pd.DataFrame):
    if "hour" not in df.columns:
        raise KeyError("Missing 'hour'. Run feature_engineering(df) first.")

    if "is_weekend" not in df.columns:
        raise KeyError("Missing 'is_weekend'. Run feature_engineering(df) first.")

    # 1. Traffic by hour
    hourly_avg = df.groupby("hour")["traffic_volume"].mean()

    plt.figure(figsize=(10,5))
    plt.plot(hourly_avg.index, hourly_avg.values, marker="o")
    plt.title("Average Traffic Volume by Hour")
    plt.xlabel("Hour")
    plt.ylabel("Traffic Volume")
    plt.grid(True)
    plt.savefig("traffic_by_hour.png")
    plt.close()
    logging.info("Saved figure: traffic_by_hour.png")
    print("Interpretation: Traffic peaks during morning and evening commute hours.")

    # 2. Weekday vs Weekend
    weekday_avg = df.groupby("is_weekend")["traffic_volume"].mean()

    plt.figure(figsize=(8,5))
    plt.bar(["Weekday", "Weekend"], weekday_avg.values)
    plt.title("Weekday vs Weekend Traffic")
    plt.ylabel("Traffic Volume")
    plt.savefig("weekday_vs_weekend.png")
    plt.close()
    logging.info("Saved figure: weekday_vs_weekend.png")
    print("Interpretation: Weekday traffic is higher due to commuting.")

    # 3. Temperature vs Traffic
    plt.figure(figsize=(10,5))
    plt.scatter(df["temp"], df["traffic_volume"], alpha=0.3)
    plt.title("Temperature vs Traffic Volume")
    plt.xlabel("Temperature")
    plt.ylabel("Traffic Volume")
    plt.savefig("temperature_vs_traffic.png")
    plt.close()
    logging.info("Saved figure: temperature_vs_traffic.png")
    print("Interpretation: Temperature has weak correlation with traffic volume.")

In [16]:
df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")

df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
df = df.dropna(subset=["date_time"])

df = feature_engineering(df)

visualise_traffic_patterns(df)

INFO:root:Feature Engineering START — Shape before: (48204, 9)
INFO:root:Feature Engineering END — Shape after: (48204, 21)
INFO:root:Saved figure: traffic_by_hour.png
INFO:root:Saved figure: weekday_vs_weekend.png


Interpretation: Traffic peaks during morning and evening commute hours.
Interpretation: Weekday traffic is higher due to commuting.


INFO:root:Saved figure: temperature_vs_traffic.png


Interpretation: Temperature has weak correlation with traffic volume.


In [17]:
print("hour" in df.columns)
print("is_weekend" in df.columns)
print("temp" in df.columns)
print("traffic_volume" in df.columns)

True
True
True
True


In [18]:
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)

def run_traffic_app(df):
    """
    Mini command-line traffic analytics application.
    Supports multiple commands with logging and error handling.
    """

    print("\n=== Mini Traffic Analytics Application ===")
    print("Available commands:")
    print("1. traffic_at <YYYY-MM-DD HH:MM>")
    print("2. high_traffic_periods")
    print("3. compare_weekday_weekend")
    print("4. recommended_travel")
    print("5. weather_traffic_at <YYYY-MM-DD HH:MM>")
    print("Type 'exit' to quit.\n")

    while True:
        user_input = input("Enter command: ").strip()

        if user_input.lower() == "exit":
            print("Exiting application.")
            break

        # ---------------------------------------------------------
        # COMMAND 1 — Query traffic for a specific date/time
        # ---------------------------------------------------------
        if user_input.startswith("traffic_at"):
            try:
                logging.info(f"Command invoked: {user_input}")

                _, dt_str = user_input.split(" ", 1)

                try:
                    dt = datetime.strptime(dt_str, "%Y-%m-%d %H:%M")
                except ValueError:
                    logging.error("ERROR: Invalid date format. Use YYYY-MM-DD HH:MM")
                    print("Invalid date format. Use YYYY-MM-DD HH:MM")
                    continue

                row = df[df["date_time"] == dt]

                if row.empty:
                    print("No data found for that date/time.")
                else:
                    vol = row.iloc[0]["traffic_volume"]
                    print(f"Traffic volume at {dt}: {vol}")

            except Exception as e:
                logging.error(f"ERROR: {str(e)}")
                print("An error occurred. Please try again.")

        # ---------------------------------------------------------
        # COMMAND 2 — Identify high-traffic periods
        # ---------------------------------------------------------
        elif user_input == "high_traffic_periods":
            logging.info("Command invoked: high_traffic_periods")

            high = df[df["congestion_category"] == "high"]
            result = high.groupby("hour")["traffic_volume"].mean().sort_values(ascending=False)

            print("\nHigh Traffic Hours (sorted):")
            print(result.head(10))

        # ---------------------------------------------------------
        # COMMAND 3 — Compare weekday vs weekend traffic
        # ---------------------------------------------------------
        elif user_input == "compare_weekday_weekend":
            logging.info("Command invoked: compare_weekday_weekend")

            weekday_avg = df[df["is_weekend"] == 0]["traffic_volume"].mean()
            weekend_avg = df[df["is_weekend"] == 1]["traffic_volume"].mean()

            print(f"\nAverage Weekday Traffic: {weekday_avg:.2f}")
            print(f"Average Weekend Traffic: {weekend_avg:.2f}")

        # ---------------------------------------------------------
        # COMMAND 4 — Recommended travel periods (lowest traffic)
        # ---------------------------------------------------------
        elif user_input == "recommended_travel":
            logging.info("Command invoked: recommended_travel")

            low = df[df["congestion_category"] == "low"]
            result = low.groupby("hour")["traffic_volume"].mean().sort_values()

            print("\nRecommended Travel Hours (lowest traffic):")
            print(result.head(10))

        # ---------------------------------------------------------
        # COMMAND 5 — Retrieve weather + traffic at a specific time
        # ---------------------------------------------------------
        elif user_input.startswith("weather_traffic_at"):
            try:
                logging.info(f"Command invoked: {user_input}")

                _, dt_str = user_input.split(" ", 1)

                try:
                    dt = datetime.strptime(dt_str, "%Y-%m-%d %H:%M")
                except ValueError:
                    logging.error("ERROR: Invalid date format. Use YYYY-MM-DD HH:MM")
                    print("Invalid date format. Use YYYY-MM-DD HH:MM")
                    continue

                row = df[df["date_time"] == dt]

                if row.empty:
                    print("No data found for that date/time.")
                else:
                    r = row.iloc[0]
                    print(f"\nWeather & Traffic at {dt}:")
                    print(f"Traffic Volume: {r['traffic_volume']}")
                    print(f"Weather: {r['weather_main']}")
                    print(f"Temperature: {r['temp']} K")
                    print(f"Cloud Cover: {r['clouds_all']}%")

            except Exception as e:
                logging.error(f"ERROR: {str(e)}")
                print("An error occurred. Please try again.")

        # ---------------------------------------------------------
        # INVALID COMMAND
        # ---------------------------------------------------------
        else:
            logging.error(f"ERROR: Unknown command '{user_input}'")
            print("Unknown command. Please try again.")

In [ ]:
df = feature_engineering(df)
run_traffic_app(df)

INFO:root:Feature Engineering START — Shape before: (48204, 21)
INFO:root:Feature Engineering END — Shape after: (48204, 21)



=== Mini Traffic Analytics Application ===
Available commands:
1. traffic_at <YYYY-MM-DD HH:MM>
2. high_traffic_periods
3. compare_weekday_weekend
4. recommended_travel
5. weather_traffic_at <YYYY-MM-DD HH:MM>
Type 'exit' to quit.

